In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import linear_kernel
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Load the dataset
zomato_raw = pd.read_csv("zomato.csv")

# Checking dataset size
print(zomato_raw.shape)

# Checking the columns in the dataset
print(zomato_raw.columns)

# Display first two rows
zomato_raw.head(2)

In [ ]:
# Keep essential columns
zomato_df = zomato_raw[['name', 'reviews_list', 'cuisines', 'rate', 'approx_cost(for two people)']].copy()
zomato_df.rename(columns={'rate': 'Mean Rating', 'approx_cost(for two people)': 'cost'}, inplace=True)

# Clean Rating
def clean_rate(value):
    if pd.isnull(value) or value == 'NEW' or value == '-': return 0.0
    return float(str(value).split('/')[0].strip())
zomato_df['Mean Rating'] = zomato_df['Mean Rating'].apply(clean_rate)

# Clean Cost
def clean_cost(value):
    if pd.isnull(value): return 0.0
    return float(str(value).replace(',', '').strip())
zomato_df['cost'] = zomato_df['cost'].apply(clean_cost)

# Drop duplicates and null reviews
zomato_df.drop_duplicates(subset='name', keep='first', inplace=True)
zomato_df.dropna(subset=['reviews_list'], inplace=True)

print("Data Cleaning Complete. Unique Restaurants:", len(zomato_df))

In [ ]:
# 1. Most Famous 6 restaurants in Bangalore
plt.figure(figsize=(10,7))
chains = zomato_df['name'].value_counts()[:6]
sns.barplot(x=chains.index, y=chains, palette='tab10')
plt.title("Most famous restaurants in Bangalore")
plt.ylabel("Number of outlets")
plt.show()

# 2. Distribution of Restaurant Rating
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 5))
sns.histplot(zomato_df['Mean Rating'], kde=False, color='b', ax=ax, bins=20)
ax.axvline(zomato_df['Mean Rating'].mean(), 0, 1, color='r', label='Mean')
ax.legend()
ax.set_ylabel('Count', size=20)
ax.set_xlabel('Rate', size=20)
ax.set_title('Distribution(count) of Restaurant rating', size=20)
plt.show()

# 3. Top 10 Rated Restaurants
df_rating = zomato_df.sort_values(by='Mean Rating', ascending=False).head(10)
plt.figure(figsize=(10,7))
sns.barplot(data=df_rating, x='Mean Rating', y='name', palette='RdBu')
plt.title('Top Rated 10 Restaurants')
plt.show()

In [ ]:
# Prepare index
df_percent = zomato_df.copy()
df_percent.set_index('name', inplace=True)
indices = pd.Series(df_percent.index)

# Creating tf-idf matrix
tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=0.0, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_percent['reviews_list'].fillna(''))

# Calculating Cosine Similarity
cosine_similarities = linear_kernel(tfidf_matrix, tfidf_matrix)
print("TF-IDF Matrix and Cosine Similarities generated successfully.")

In [ ]:
def recommend(name, cosine_similarities=cosine_similarities):
    recommend_restaurant = []
    
    if name not in indices.values:
        return "Restaurant not found."

    idx = indices[indices == name].index[0]
    score_series = pd.Series(cosine_similarities[idx]).sort_values(ascending=False)
    top_indexes = list(score_series.iloc[1:min(31, len(score_series))].index)
    
    for each in top_indexes:
        recommend_restaurant.append(list(df_percent.index)[each])
    
    df_new = pd.DataFrame(columns=['cuisines', 'Mean Rating', 'cost'])
    
    for each in recommend_restaurant:
        selected_data = df_percent[['cuisines', 'Mean Rating', 'cost']][df_percent.index == each]
        if not selected_data.empty:
            df_new = pd.concat([df_new, selected_data.sample(n=1)])
    
    df_new = df_new.drop_duplicates(subset=['cuisines', 'Mean Rating', 'cost'], keep=False)
    df_new = df_new.sort_values(by='Mean Rating', ascending=False).head(10)
    
    print(f"TOP 10 RESTAURANTS LIKE {name} WITH SIMILAR REVIEWS: \n")
    return df_new

# Testing the recommendations exactly as requested in the project manual
display(recommend('Red Chilliez'))
display(recommend('Cinnamon'))
display(recommend('Spice Up'))
display(recommend('Desi Doze'))

In [ ]:
# Exporting the cleaned dataframe and the trained model for the web application
zomato_df.to_csv("restaurant1.csv", index=False)

with open("restaurant.pkl", "wb") as f:
    pickle.dump(cosine_similarities, f)
    
print("Files exported successfully!")